# Silver Exploration Notebook


## Imports

In [10]:
import os
import findspark

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    avg,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    split,
    element_at,
    countDistinct,
    sum as spark_sum
)

## Start Spark

In [11]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 14:34:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

/home/ubuntu/spark-lab-data/dev/project/raw


## Read Bike-sharing dataset
+ Adding city and month

In [4]:
# Bike-sharing data
trips_raw = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
    .withColumn("src", input_file_name())
)

trips_raw = (
    trips_raw
    .withColumn("city", element_at(split(col("src"), "/"), -2))
    .withColumn("year_month", element_at(split(col("src"), "/"), -1))
    .withColumn("year_month",split(col("year_month"), r"\.").getItem(0))
)

trips_raw.select("city", "year_month").show(5)

+----+----------+
|city|year_month|
+----+----------+
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
+----+----------+
only showing top 5 rows



## Exporation of Bike-sharing dataset

In [5]:
trips_raw.printSchema()
trips_raw.show(5, truncate=False)
print("Total rows:", trips_raw.count())
print("Columns:", trips_raw.columns)

root
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: string (nullable = true)
 |-- start_station_longitude: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: string (nullable = true)
 |-- end_station_longitude: string (nullable = true)
 |-- src: string (nullable = false)
 |-- city: string (nullable = true)
 |-- year_month: string (nullable = true)

+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+-----------

Total rows: 8470330
Columns: ['started_at', 'ended_at', 'duration', 'start_station_id', 'start_station_name', 'start_station_description', 'start_station_latitude', 'start_station_longitude', 'end_station_id', 'end_station_name', 'end_station_description', 'end_station_latitude', 'end_station_longitude', 'src', 'city', 'year_month']


- We have more than 8 470 000 biketrips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `endend_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. We also have the added `src`, `city` and `year_month`.

#### Data Types
The 16 columns are stored as string data types. This is because PySpark reads CSV columns as strings by default. However, Several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> integer(maybe double?)
- Coorinates: `latitude`and `longitude`-> double

Maybe:
- StationIDs: `start_station_id` and `stop_station_id` -> keep as string or convert to integer

From the table above we can see the ID is stored as a numeric value, but we would argue to keep them as a string as they are identifiers rather than numeric measurements. This also provides a greater flexibility if future datasets contains letters(e.g. ST123), without requiring changes to the data type. 


### Investigating missing values

In [6]:
trips_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips_raw.columns
]).show()

+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+
|started_at|ended_at|duration|start_station_id|start_station_name|start_station_description|start_station_latitude|start_station_longitude|end_station_id|end_station_name|end_station_description|end_station_latitude|end_station_longitude|src|city|year_month|
+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+
|         0|       0|       0|               0|                 0|                    11170|                     0|                      0|             0|               0|                   7112|                   0|       

#### Missing values
The `start_station_description` column contains 11 000 missing values, while `end_station_description` contains around 7000 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysing we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station ID and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to are final analytical dataset. 

### Investigating duplicates

In [7]:
print("Total rows:", trips_raw.count())
print("Duplicate rows:", trips_raw.count() - trips_raw.distinct().count())

Total rows: 8470330


26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:33:10 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate rows: 25


Out of the 8 470 330 trips, 25 additional duplicates. 

#### Are we sure these are duplicates?

We can not be certain that these are dublicates because the trips do not contain any identifier. These records do however contain identical entries across all 16 collumns, including timestamp with microseconds precision. It is highly unlikely that separate trips would share all these values.

We consider therefore these records to be duplicate entries, potentially from the source data or overlapping records during the ingestion process. The exact cause has not been investigated further, but we want to remove the additional copies. 

Given that our dataset contain over 8 million trips, removing 25 duplicate records will have a negligible impact on the overall analysis. If the dataset was significanlty smaller, further investigation would be important as removing even a small number of records could have a greater influence on the result.

In [8]:
duplicate_cols = [
    c for c in trips_raw.columns
    if c not in ["src", "year_month"]
]

trips_silver = trips_raw.dropDuplicates(duplicate_cols)

print("Total rows:", trips_raw.count())
print("Duplicate rows:", trips_raw.count() - trips_silver.count())

Total rows: 8470330


26/11/15 12:34:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:34:16 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate rows: 25


### Investigatin values and distributions

#### First Duration
Lets start with convert the `duration` column to a double data type. 

In [9]:
trips_silver = (
    trips_silver
    .withColumn("duration", col("duration").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("duration").isNull() 
).count()


0

In [10]:

trips_silver.select("duration").summary("min", "25%", "50%", "75%", "max").show()

trips_silver.select(spark_sum(when(col("duration") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration").isNull(), 1).otherwise(0)).alias("null_duration")).show()

26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:35:32 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+--------+
|summary|duration|
+-------+--------+
|    min|     0.0|
|    25%|   340.0|
|    50%|   524.0|
|    75%|   805.0|
|    max| 79790.0|
+-------+--------+



26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:36:21 WARN RowBasedKeyValueBatch: Calling spill() on

+------------+---------+-------------+
|non_positive|under_60s|null_duration|
+------------+---------+-------------+
|           2|     1310|            0|
+------------+---------+-------------+



#### Durations to investigate ferther
- 2 trips has a negative or a duration of 0.
- Is it possible that a trip last around 22 hours? (79790/3600 = 22)

In [11]:
# Trips with durations smaller than 1 second
trips_silver.filter(
    col("duration").cast("double") <= 1
).show(truncate=False)

print("How many trips have a duration of 1 second or less?", trips_silver.filter(col("duration").cast("double") <= 1).count())

+--------------------------------+--------------------------------+--------+----------------+-------------------+------------------------------------------------------+----------------------+-----------------------+--------------+-------------------------+-----------------------------------------------------+--------------------+---------------------+------------------------------------------------------------------------------+---------+----------+
|started_at                      |ended_at                        |duration|start_station_id|start_station_name |start_station_description                             |start_station_latitude|start_station_longitude|end_station_id|end_station_name         |end_station_description                              |end_station_latitude|end_station_longitude|src                                                                           |city     |year_month|
+--------------------------------+--------------------------------+--------+----------------

How many trips have a duration of 1 second or less? 29


### Extremely Short Trips
29 trips with a recorded duration of one second or less, including two trips with a duration of zero seconds. 

With the table above these trips have a different start and end station which makes it physically impossible to complete the journeys within that timeframe.

These records likely represent data quality issues that need futher investigation.

Possible solution: 
- Find the mean trip duration from one statiotn to the other and compare it to minimum duration record.
- Flag trips under a spesific timeframe. 


In [12]:
station_durations = (
    trips_silver
    .groupBy("city", "start_station_id", "end_station_id")
    .agg(
        count("*").alias("num_trips"),
        avg("duration").alias("mean_duration"),
        min("duration").alias("min_duration")
    )
)

station_durations.filter(
    col("num_trips") >= 10
).orderBy("min_duration").show(20, truncate=False)

26/11/15 12:37:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:27 WARN RowBasedKeyValueBatch: Calling spill() on

26/11/15 12:37:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:37:58 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------------+--------------+---------+------------------+------------+
|city     |start_station_id|end_station_id|num_trips|mean_duration     |min_duration|
+---------+----------------+--------------+---------+------------------+------------+
|oslo     |465             |523           |93       |585.3978494623656 |0.0         |
|bergen   |217             |815           |373      |241.01072386058982|0.0         |
|trondheim|10              |293           |201      |274.318407960199  |1.0         |
|trondheim|14              |45            |232      |413.48275862068965|1.0         |
|trondheim|1878            |18            |165      |641.6242424242424 |1.0         |
|bergen   |2336            |807           |194      |1109.7628865979382|1.0         |
|bergen   |788             |808           |1001     |409.4725274725275 |1.0         |
|trondheim|155             |1882          |303      |720.1221122112212 |1.0         |
|trondheim|25              |1             |92       |1

### What does the table above tell us?
Although the mean duration does not accually tell us the minimum possible duration between two stations, it provides a useful indication of what a typical trip looks like. 

The table shows station pairs with hundres and thousands of bike-trips, where the average duration is several minutes, yet the shortest recorded time is only one second. This strongly suggest that some values are incorrect or unrealistic. 

On another note this does make me question the dataset and its accuracy. Are these simply a small number of isolated data quality issues, or could they indicate more problems with the data?

For now let's atleast flag trips with a duration smaller than one second. 

In [13]:
# Flag trips that are under 1 second
trips_silver = trips_silver.withColumn(
    "short_trip_flag",
    col("duration") <= 1
)

### Trips with a long duration

In [14]:
print("Number of trips longer than 12 hours: ", trips_silver.filter(col("duration") > 12 * 3600).count())

trips_silver.select(
    "started_at",
    "ended_at",
    "duration"
).orderBy(desc("duration")).show(15, truncate=False)

Number of trips longer than 12 hours:  15


26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:38:25 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------------------+--------------------------------+--------+
|started_at                      |ended_at                        |duration|
+--------------------------------+--------------------------------+--------+
|2026-06-19 13:38:13.233000+00:00|2026-06-20 11:48:03.076000+00:00|79790.0 |
|2025-10-17 14:35:20.232000+00:00|2025-10-18 11:15:52.091000+00:00|74431.0 |
|2022-04-16 12:47:44.363000+00:00|2022-04-17 07:37:10.178000+00:00|67765.0 |
|2023-04-07 13:40:11.612000+00:00|2023-04-08 08:18:37.147000+00:00|67105.0 |
|2023-05-31 15:31:03.704000+00:00|2023-06-01 08:37:01.096000+00:00|61557.0 |
|2026-05-04 18:28:16.356000+00:00|2026-05-05 11:14:23.840000+00:00|60367.0 |
|2022-11-01 15:19:05.897000+00:00|2022-11-02 07:26:44.435000+00:00|58058.0 |
|2022-11-24 14:20:12.725000+00:00|2022-11-25 05:44:40.784000+00:00|55468.0 |
|2026-08-06 20:38:28.625000+00:00|2026-08-07 11:37:01.191000+00:00|53913.0 |
|2022-05-20 16:49:31.692000+00:00|2022-05-21 06:44:45.835000+00:00|50114.0 |

Could a city bike realistically be used for appocimately 22 hours?

Accoring to Oslo Bysykkel, a fully charged electric bike has an estimate range of around 70 kilometers. The battery does not necessarily limit the recorded trip duration, as the bike can remain parked while the rental is still active and you can manually ride the bike if the battery runs out. 

If the dataset included distance travelled we could better investigate whether the bicycle was actively used.

Possible explainations for the long duration
- The user forgot to return the bike or didn't properly complete the return process.
- The bike was parked for several hours while the rental was still active. 
- The recorded timestamp contains an error. 
- Maybe others?

In [15]:
print("Over 3 hours:", trips_silver.filter(col("duration") > 3 * 60 * 60).count())
print("Over 6 hours:", trips_silver.filter(col("duration") > 6 * 60 * 60).count())
print("Over 12 hours:", trips_silver.filter(col("duration") > 12 * 60 * 60).count())

Over 3 hours: 3184


Over 6 hours: 315


Over 12 hours: 15


In [16]:
total_trips = trips_silver.count()

long_trips = trips_silver.filter(
    col("duration") > 3 * 3600 
).count()

print(f"{long_trips / total_trips * 100}%")

26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:39:39 WARN RowBasedKeyValueBatch: Calling spill() on

0.037590145809389394%


(3184/8470330) * 100 = 0.0376 %

Only 0.038% of all bike trips last longer than 3 hours, inducating such trips are extremely rare.

### Investigating the timestamp format

In [17]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:35 WARN RowBasedKeyValueBatch: Calling spill() on

26/11/15 12:40:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:40:56 WARN RowBasedKeyValueBatch: Calling spill() on

+---+-------+
|len|  count|
+---+-------+
| 25|   8226|
| 32|8462079|
+---+-------+



26/11/15 12:41:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------------------------+--------------------------------+--------+
|started_at                      |ended_at                        |duration|
+--------------------------------+--------------------------------+--------+
|2022-08-01 05:35:31.981000+00:00|2022-08-01 05:51:32.968000+00:00|960.0   |
|2022-08-01 05:39:23.445000+00:00|2022-08-01 05:54:46.326000+00:00|922.0   |
|2022-08-01 07:28:41.156000+00:00|2022-08-01 07:40:31.128000+00:00|709.0   |
|2022-08-01 07:33:41.640000+00:00|2022-08-01 07:43:22.460000+00:00|580.0   |
|2022-08-01 09:31:48.451000+00:00|2022-08-01 09:41:56.865000+00:00|608.0   |
|2022-08-01 10:25:16.864000+00:00|2022-08-01 10:29:04.710000+00:00|227.0   |
|2022-08-01 10:34:29.583000+00:00|2022-08-01 10:49:44.016000+00:00|914.0   |
|2022-08-01 13:36:38.583000+00:00|2022-08-01 13:43:00.941000+00:00|382.0   |
|2022-08-01 14:31:37.484000+00:00|2022-08-01 14:40:59.374000+00:00|561.0   |
|2022-08-01 14:35:07.091000+00:00|2022-08-01 14:45:50.809000+00:00|643.0   |

#### Converting the string to UTC timestamp
The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

In [18]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips_silver = (
    trips_silver
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Let's compare the converted timestamps
trips_silver.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:41:32 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------------------+-----------------------+--------------------------------+-----------------------+
|started_at                      |started_ts             |ended_at                        |ended_ts               |
+--------------------------------+-----------------------+--------------------------------+-----------------------+
|2022-08-01 04:45:18.184000+00:00|2022-08-01 04:45:18.184|2022-08-01 04:54:05.085000+00:00|2022-08-01 04:54:05.085|
|2022-08-01 05:52:07.970000+00:00|2022-08-01 05:52:07.97 |2022-08-01 06:00:58.168000+00:00|2022-08-01 06:00:58.168|
|2022-08-01 06:39:46.021000+00:00|2022-08-01 06:39:46.021|2022-08-01 06:51:09.655000+00:00|2022-08-01 06:51:09.655|
|2022-08-01 06:44:33.741000+00:00|2022-08-01 06:44:33.741|2022-08-01 06:51:40.306000+00:00|2022-08-01 06:51:40.306|
|2022-08-01 11:34:21.421000+00:00|2022-08-01 11:34:21.421|2022-08-01 11:41:26.002000+00:00|2022-08-01 11:41:26.002|
|2022-08-01 14:15:51.966000+00:00|2022-08-01 14:15:51.966|2022-08-01 14:

In [19]:
# Did the convertion work?
trips_silver.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

0

### Converting Timestamps to Norwegian Local Time

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [20]:
trips_silver = (
    trips_silver
    .withColumn(
        "started_local",
        from_utc_timestamp("started_ts", "Europe/Oslo")
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp("ended_ts", "Europe/Oslo")
    )
    
)

trips_silver.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:24 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------------------+-----------------------+-----------------------+-----------------------+
|started_ts             |started_local          |ended_ts               |ended_local            |
+-----------------------+-----------------------+-----------------------+-----------------------+
|2022-08-01 04:45:18.184|2022-08-01 06:45:18.184|2022-08-01 04:54:05.085|2022-08-01 06:54:05.085|
|2022-08-01 05:52:07.97 |2022-08-01 07:52:07.97 |2022-08-01 06:00:58.168|2022-08-01 08:00:58.168|
|2022-08-01 06:39:46.021|2022-08-01 08:39:46.021|2022-08-01 06:51:09.655|2022-08-01 08:51:09.655|
|2022-08-01 06:44:33.741|2022-08-01 08:44:33.741|2022-08-01 06:51:40.306|2022-08-01 08:51:40.306|
|2022-08-01 11:34:21.421|2022-08-01 13:34:21.421|2022-08-01 11:41:26.002|2022-08-01 13:41:26.002|
|2022-08-01 14:15:51.966|2022-08-01 16:15:51.966|2022-08-01 14:33:13.555|2022-08-01 16:33:13.555|
|2022-08-01 14:55:11.433|2022-08-01 16:55:11.433|2022-08-01 15:19:08.435|2022-08-01 17:19:08.435|
|2022-08-01 15:07:54

#### Does duration correspond to the start and end time?

In [21]:
trips_silver.select(
    "started_local",
    "ended_local",
    "duration",
    # does started_at and ended_at match the duration?
    (
        (
            unix_micros("ended_ts") -
            unix_micros("started_ts")
        ) / 1000000
    ).alias("duration_check")

).show(5, truncate=False)

26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:42:59 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------------------+-----------------------+--------+--------------+
|started_local          |ended_local            |duration|duration_check|
+-----------------------+-----------------------+--------+--------------+
|2022-08-01 06:45:18.184|2022-08-01 06:54:05.085|526.0   |526.901       |
|2022-08-01 07:52:07.97 |2022-08-01 08:00:58.168|530.0   |530.198       |
|2022-08-01 08:39:46.021|2022-08-01 08:51:09.655|683.0   |683.634       |
|2022-08-01 08:44:33.741|2022-08-01 08:51:40.306|426.0   |426.565       |
|2022-08-01 13:34:21.421|2022-08-01 13:41:26.002|424.0   |424.581       |
+-----------------------+-----------------------+--------+--------------+
only showing top 5 rows




From the table above it looks like the duration always round down.

for example:
- 363 and 363.712
- 703 and 703
- 778 and 778.756

Let's investigate the duration a bit further.

In [22]:
trips_check = trips_silver.withColumn(
    "duration_diff",
    abs(
        (unix_micros("ended_ts")
         - unix_micros("started_ts")) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:43:34 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+-------------------+
|summary|      duration_diff|
+-------+-------------------+
|  count|            8470305|
|   mean| 0.4555726149176461|
| stddev|0.28521666537753404|
|    min|                0.0|
|    25%|0.21199999999998909|
|    50%|0.42500000000001137|
|    75%| 0.6960000000000264|
|    max| 0.9990000000000236|
+-------+-------------------+



Since the maximum difference are below one second, we consider the recorded durations consistent with the timestamps. 

### Investigating latitude and longitude
first converting the data types to double

In [23]:
trips_silver = (
    trips_silver
    .withColumn("start_station_latitude", col("start_station_latitude").cast("double"))
    .withColumn("start_station_longitude", col("start_station_longitude").cast("double"))
    .withColumn("end_station_latitude", col("end_station_latitude").cast("double"))
    .withColumn("end_station_longitude", col("end_station_longitude").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("start_station_latitude").isNull() |
    col("start_station_longitude").isNull() |
    col("end_station_latitude").isNull() |
    col("end_station_longitude").isNull()
).count()

0

#### Valid cordination for latitude and longitude in Norway:
- Norwegian latitude: 57 - 71
- Norwegian longitude: 4 - 31

In [24]:
trips_silver.filter(
    (col("start_station_latitude") < 57) |
    (col("start_station_latitude") > 71) |
    (col("end_station_latitude") < 57) |
    (col("end_station_latitude") > 71) |
    (col("start_station_longitude") < 4) |
    (col("start_station_longitude") > 31) |
    (col("end_station_longitude") < 4) |
    (col("end_station_longitude") > 31)
).count()

0

In [25]:
trips_silver.select(
    min("start_station_latitude").alias("min_lat"),
    max("start_station_latitude").alias("max_lat"),
    min("start_station_longitude").alias("min_lon"),
    max("start_station_longitude").alias("max_lon")
).show()

+-----------------+------------------+-----------------+-----------------+
|          min_lat|           max_lat|          min_lon|          max_lon|
+-----------------+------------------+-----------------+-----------------+
|59.89843428245928|63.446059058709096|5.273174034250616|10.81431433498766|
+-----------------+------------------+-----------------+-----------------+



The latitude and logitude values appear to reasonable based on the geographical locations of the three cities. Oslo is the city most southern, while Trondheim is the city most nothern. Bergen is the city most western, while oslo is the city most eastern. 

|   City       |   latitude    |   longitude   |
|--------------|---------------|---------------|
|  Oslo:       |     59.91     |     10.75     |  
|  Bergen:     |     60.39     |      5.32     |
|  Trondheim:  |     63.43     |     10.40     |

### Number of trips divided by the different cities

In [26]:
trips_silver.groupBy("city").count().show()

26/11/15 12:44:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:44:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:44:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:44:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:44:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:44:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:01 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+-------+
|     city|  count|
+---------+-------+
|trondheim| 815111|
|     oslo|5750607|
|   bergen|1904587|
+---------+-------+



Oslo accounts for approximately 68% of all trips in the dataset, so the analysis of the trips may be dominated by the travel patterns in Oslo.

#### Number of trips devided by city and stations

In [27]:
trips_silver.groupBy("city", "start_station_id", "end_station_id").count().show(200)

26/11/15 12:45:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:45:45 WARN RowBasedKeyValueBatch: Calling spill() on

+---------+----------------+--------------+-----+
|     city|start_station_id|end_station_id|count|
+---------+----------------+--------------+-----+
|     oslo|             529|           412|   35|
|     oslo|             449|           970|   17|
|     oslo|             426|           455|   18|
|     oslo|             408|           970|  741|
|     oslo|            2306|           411|  145|
|     oslo|             478|           523|  175|
|     oslo|             524|           453|  124|
|     oslo|             441|           464|  216|
|     oslo|            1101|           579|   33|
|     oslo|             442|           611|  452|
|     oslo|             609|           503|  113|
|     oslo|             412|           479|  184|
|     oslo|             579|           475|   28|
|     oslo|             455|           533|   32|
|     oslo|             597|          2315|  224|
|     oslo|             586|           607|  484|
|     oslo|             608|           540|  156|


From the table above it looks like the stationIDs might use the same identifications accross the cities. So the ID alone won't be possible to identify which city the trip was in.  

Let's investigate this a bit ferther:

In [28]:
trips_silver.groupBy("start_station_id").agg(
    countDistinct("city").alias("num_cities")
).filter(
    col("num_cities") > 1
).show()

+----------------+----------+
|start_station_id|num_cities|
+----------------+----------+
+----------------+----------+



The investigantion shows that no start station id is shared across cities in this current dataset. I do think including cities when grouping by station IDs to ensure robustness against potential ovelaps in future data and it gives a better overview. 

#### Number of trips divided by city and month

In [29]:
trips_silver.groupBy("city", "year_month").count().orderBy(
    "city", "year_month"
).show(200)

+---------+----------+------+
|     city|year_month| count|
+---------+----------+------+
|   bergen|   2022-01| 26375|
|   bergen|   2022-02| 28389|
|   bergen|   2022-03| 50234|
|   bergen|   2022-04| 49972|
|   bergen|   2022-05| 56259|
|   bergen|   2022-06| 57101|
|   bergen|   2022-07| 30356|
|   bergen|   2022-08| 60298|
|   bergen|   2022-09| 68782|
|   bergen|   2022-10| 47233|
|   bergen|   2022-11| 42541|
|   bergen|   2022-12| 17773|
|   bergen|   2023-01| 24829|
|   bergen|   2023-02| 30059|
|   bergen|   2023-03| 30451|
|   bergen|   2023-04| 41145|
|   bergen|   2023-05| 49749|
|   bergen|   2023-06| 58606|
|   bergen|   2023-07| 35019|
|   bergen|   2023-08| 52498|
|   bergen|   2023-09| 49709|
|   bergen|   2023-10| 42989|
|   bergen|   2023-11| 36744|
|   bergen|   2023-12| 16062|
|   bergen|   2024-01| 20408|
|   bergen|   2024-02| 19706|
|   bergen|   2024-03| 29804|
|   bergen|   2024-04| 43104|
|   bergen|   2024-05| 56798|
|   bergen|   2024-06| 42879|
|   bergen

### Summary and proposed Silver tranformations
Based on this explorations analysis, this is the following transformations we propose for the Silver layer:
- Extract city and year-month information from the source file path.
- Convert trip duration and geographical coordinates to double. 
- Converting start and and timestamps to UTC timestamps.
- Derive Norwegian local timestamps for temporal analysis. 
- Remove duplicate trips records. 
- Perserve station IDs as strings. 
- Perserve missing station descriptions, as they do not affect the main analytical objective.
- Flag trips with a recorded duration of one second. (Maybe even more?)
- Validate recorded durations against the start and end timestamps. 
- Validate geographical coordinates.

### Test silver_trips.py

In [7]:
import sys
import os

print(os.getcwd)
sys.path.append("../Silver")

from silver_trips import (
    read_raw_trips,
    build_silver_trips,
    trips_quality_metrics
)

<built-in function getcwd>


In [14]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

trips_silver_test = build_silver_trips(spark, RAW)

trips_silver_test.printSchema()
trips_silver_test.show(5, truncate=False)

root
 |-- duration: double (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_longitude: double (nullable = true)
 |-- city: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- started_ts: timestamp (nullable = true)
 |-- ended_ts: timestamp (nullable = true)
 |-- started_local: timestamp (nullable = true)
 |-- ended_local: timestamp (nullable = true)
 |-- short_trip_flag: boolean (nullable = true)
 |-- duration_diff: double (nullable = true)
 |-- invalid_duration_flag: boolean (nullable = true)
 |-- invalid_coordinates_flag:

+--------+----------------+--------------------+-----------------------------------------------+----------------------+-----------------------+--------------+-----------------+-------------------------------------------+--------------------+---------------------+---------+----------+-----------------------+-----------------------+-----------------------+-----------------------+---------------+-------------------+---------------------+------------------------+-----------------------+
|duration|start_station_id|start_station_name  |start_station_description                      |start_station_latitude|start_station_longitude|end_station_id|end_station_name |end_station_description                    |end_station_latitude|end_station_longitude|city     |year_month|started_ts             |ended_ts               |started_local          |ended_local            |short_trip_flag|duration_diff      |invalid_duration_flag|invalid_coordinates_flag|_silver_processed_time |
+--------+--------------

In [15]:
trips_quality_metrics(spark, RAW)

26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 14:37:36 WARN RowBasedKeyValueBatch: Calling spill() on

{'layer': 'silver_trips',
 'rows_in': 8470330,
 'rows_out': 8470305,
 'duplicates_removed': 25,
 'short_trips': 29,
 'invalid_duration': 0,
 'invalid_coordinates': 0}